# Prep ParlaSpeech — chapter 1

Parse a ParlaSpeech-{LANG} JSONL into the canonical pipeline JSONL. Three outputs from one pass:

1. **`parlaspeech_{lang}_instance.jsonl`** — one record per utterance OR per filled-pause event (see `instance_mode`).
2. **`parlaspeech_{lang}_frame.jsonl`** — one record per utterance with a **50 Hz frame label sequence**.
3. **WAVs** in `data/cut_audio/ParlaSpeech-{LANG}/` — FLACs converted from the audio release.

**Inputs**
- Annotations: `data/unpacked/ParlaSpeech-{LANG}/ParlaSpeech-{LANG}.v3.0/ParlaSpeech-{LANG}.v3.0.jsonl`
- Audio: `data/unpacked/ParlaSpeech-{LANG}-audio/` — split across `*.part1/`, `*.part2/` etc., each containing `{hash}/{hash}_{start}-{end}.flac` files.

**Instance modes** (cell 2, `cfg.instance_mode`):

- `"utterance"` — one JSONL line per utterance. Label: `filled_pause_present` (0/1). FP intervals stored in `meta.fp_intervals` for reference.
- `"fp_event"` — one JSONL line per filled-pause event. Label always 1. `start_t`/`end_t` mark the FP within the utterance. Utterances without FPs are skipped.

**TODO**: Running in different modes overwrites the same output file. Options:
  (a) auto-name by mode (`parlaspeech_rs_instance_utterance.jsonl` / `_fp_event.jsonl`),
  (b) add a `"both"` mode that writes two files in one pass.
  Track in BLUEPRINT.md section 10 when this becomes a blocker.

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name != "1_data_prep":
    candidate = HERE / "1_data_prep"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE))

import utils_dataprep as udp
PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

---

## 1. Config

Key switches:
- `lang` — HR / RS / PL / CZ
- `instance_mode` — `"utterance"` or `"fp_event"` (see header)
- `audio_base_dir` — root of the unpacked audio release (contains part1/, part2/ dirs)
- `cleanup_flac` — delete each FLAC immediately after its WAV is written (saves disk)

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    # Language variant. One of: HR, RS, PL, CZ
    lang: str = "RS"

    # Instance output mode: "utterance" or "fp_event"  (see notebook header)
    instance_mode: str = "utterance"

    # Path to decompressed JSONL. Leave "" to auto-detect from lang.
    jsonl_path: str = ""

    # Root directory of the unpacked audio release (contains *.part1/ *.part2/ dirs).
    # Example: "data/unpacked/ParlaSpeech-RS-audio"
    # Leave "" to skip audio conversion (audio_path will be null in outputs).
    audio_base_dir: str = ""

    # Where to write the 16 kHz mono WAVs. Auto-set from lang if left "".
    output_wav_dir: str = ""

    # Delete each source FLAC immediately after its WAV is written.
    cleanup_flac: bool = False

    # Skip WAV conversion if output already exists.
    skip_existing_wavs: bool = True

    # Target sample rate for WAV output.
    target_sample_rate: int = 16000

    # Frame rate — hard-locked to 50 Hz (must match chapter 4).
    frame_rate_hz: int = 50

    # Output JSONL paths. Auto-set from lang if "".
    output_instance_jsonl: str = ""
    output_frame_jsonl:    str = ""

    # Split ratios (train / dev / test), assigned deterministically by speaker.
    split_ratios: tuple = (0.8, 0.1, 0.1)
    split_seed:   int   = 42

    # Filtering
    min_duration_s:     float = 0.1
    skip_failed_pauses: bool  = True   # drop records where filled_pauses is None

    # Test mode
    test_mode:      bool = False                                       ############ TEST MODE
    test_n_records: int  = 1000

cfg = Config()

# Auto-set paths from lang
_lang = cfg.lang
if not cfg.jsonl_path:
    cfg.jsonl_path = (
        f"data/unpacked/ParlaSpeech-{_lang}/"
        f"ParlaSpeech-{_lang}.v3.0/"
        f"ParlaSpeech-{_lang}.v3.0.jsonl"
    )
if not cfg.output_wav_dir:
    cfg.output_wav_dir = f"data/cut_audio/ParlaSpeech-{_lang}"
if not cfg.output_instance_jsonl:
    cfg.output_instance_jsonl = f"data/processed_jsonl/parlaspeech_{_lang.lower()}_instance.jsonl"
if not cfg.output_frame_jsonl:
    cfg.output_frame_jsonl = f"data/processed_jsonl/parlaspeech_{_lang.lower()}_frame.jsonl"

print(cfg)

---

## 2. Locate the JSONL

In [ ]:
jsonl_path = PROJECT_ROOT / cfg.jsonl_path

if not jsonl_path.exists():
    raise FileNotFoundError(
        f"JSONL not found at {jsonl_path}\n"
        f"Run 10_download_data.ipynb with datasets=['ParlaSpeech-{cfg.lang}'] first."
    )

size_mb = jsonl_path.stat().st_size / 1e6
print(f"✅ Found: {jsonl_path.relative_to(PROJECT_ROOT)}")
print(f"   Size:  {size_mb:.1f} MB")

---

## 3. Preflight — peek at records

In [ ]:
import json

print(f"First 3 records:\n")
with open(jsonl_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        r = json.loads(line)
        si  = r.get("speaker_info", {})
        fps = r.get("filled_pauses")
        print(f"--- Record {i} ---")
        print(f"  id:            {r['id']}")
        print(f"  audio:         {r.get('audio')}")
        print(f"  audio_length:  {r.get('audio_length', 0):.3f}s")
        print(f"  text:          {str(r.get('text',''))[:80]}")
        if fps is None:
            print(f"  filled_pauses: None  (inference failed)")
        else:
            print(f"  filled_pauses: {len(fps)} event(s)  {fps[:2]}")
        print(f"  speaker:       {si.get('Speaker_ID','?')} / {si.get('Lang','?')}")
        print()

---

## 4. Stream and parse

In [ ]:
from tqdm.auto import tqdm

records       = []
n_total       = 0
n_skip_none   = 0
n_skip_short  = 0

with open(jsonl_path, encoding="utf-8") as f:
    for line in tqdm(f, desc=f"parsing ParlaSpeech-{cfg.lang}", unit=" lines"):
        n_total += 1
        if cfg.test_mode and n_total > cfg.test_n_records:
            break

        r   = json.loads(line)
        fps = r.get("filled_pauses")
        dur = r.get("audio_length", 0)

        if cfg.skip_failed_pauses and fps is None:
            n_skip_none += 1
            continue
        if dur < cfg.min_duration_s:
            n_skip_short += 1
            continue

        records.append(r)

print(f"\nRead {n_total} lines")
print(f"  Kept:               {len(records)}")
print(f"  Skipped (None FPs): {n_skip_none}")
print(f"  Skipped (too short):{n_skip_short}")

---

## 5. Stats

In [ ]:
from collections import Counter
import statistics

fps_counts  = Counter()
durations   = []
speakers    = Counter()

for r in records:
    fps = r.get("filled_pauses") or []
    fps_counts[len(fps)] += 1
    durations.append(r.get("audio_length", 0))
    sp = r.get("speaker_info", {}).get("Speaker_ID", "unknown")
    speakers[sp] += 1

n_pos = sum(v for k, v in fps_counts.items() if k > 0)
total = len(records)

print(f"Records:             {total}")
print(f"  With filled pause: {n_pos} ({100*n_pos/total:.1f}%)")
print(f"  Without:           {total - n_pos}")
print(f"\nDuration:  mean={statistics.mean(durations):.2f}s  "
      f"median={statistics.median(durations):.2f}s  "
      f"total={sum(durations)/3600:.1f}h")
print(f"\nSpeakers:  {len(speakers)} unique")
print(f"\nFP count distribution (first 8 buckets):")
for k, v in sorted(fps_counts.items())[:8]:
    bar = "█" * min(40, int(40 * v / total))
    print(f"  {k:3d} FP(s): {v:6d}  {bar}")

---

## 6. Frame label function

50 Hz binary sequence from `filled_pauses` intervals.

In [ ]:
def compute_frame_labels(filled_pauses, audio_length, frame_rate_hz):
    """Return list of 0/1 ints, one per frame at frame_rate_hz."""
    n_frames = round(audio_length * frame_rate_hz)
    labels   = [0] * n_frames
    for fp in (filled_pauses or []):
        f_s = max(0,        round(fp["time_s"] * frame_rate_hz))
        f_e = min(n_frames, round(fp["time_e"] * frame_rate_hz))
        for i in range(f_s, f_e):
            labels[i] = 1
    return labels

# Smoke test
ex  = records[0]
seq = compute_frame_labels(ex.get("filled_pauses") or [], ex["audio_length"], cfg.frame_rate_hz)
print(f"Smoke test — {ex['id']}")
print(f"  {ex['audio_length']:.3f}s → {len(seq)} frames at {cfg.frame_rate_hz} Hz")
print(f"  FP events: {ex.get('filled_pauses')}")
print(f"  Positive frames: {sum(seq)}/{len(seq)}")

---

## 7. Split assignment

Deterministic by speaker — same speaker always in same split.

In [ ]:
import hashlib
from collections import Counter

train_r, dev_r, _ = cfg.split_ratios

def speaker_split(speaker_id: str) -> str:
    h = hashlib.md5(f"{cfg.split_seed}:{speaker_id}".encode()).hexdigest()
    v = int(h, 16) / (16 ** len(h))
    if v < train_r:
        return "train"
    elif v < train_r + dev_r:
        return "dev"
    return "test"

split_counts = Counter()
for r in records:
    sp = r.get("speaker_info", {}).get("Speaker_ID", "unknown")
    r["_split"] = speaker_split(sp)
    split_counts[r["_split"]] += 1

print("Split distribution:")
for split in ["train", "dev", "test"]:
    n = split_counts[split]
    print(f"  {split:5s}: {n:6d} ({100*n/len(records):.1f}%)")

---

## 8. Build audio index

Scan all `*.part*/` subdirs under `cfg.audio_base_dir` and build a lookup dict
`{hash/filename.flac → full Path}` in one pass.

The `audio` field in each JSONL record is a relative path like
`2iYgi9Ef6nM/2iYgi9Ef6nM_5686.7-5701.88.flac` — this becomes the lookup key.

In [ ]:
from pathlib import Path

audio_index = {}  # "hash/file.flac" -> Path to the actual FLAC

if cfg.audio_base_dir:
    audio_base = PROJECT_ROOT / cfg.audio_base_dir
    if not audio_base.exists():
        raise FileNotFoundError(
            f"audio_base_dir not found: {audio_base}\n"
            f"Run 10_download_data.ipynb with datasets=['ParlaSpeech-{cfg.lang}-audio'] first."
        )

    # Find part dirs (e.g. ParlaSpeech-RS.v1.0.part1) — or fall back to the base dir itself
    part_dirs = sorted(audio_base.glob("*.part*"))
    if not part_dirs:
        part_dirs = [audio_base]

    print(f"Scanning {len(part_dirs)} part dir(s) under {audio_base.relative_to(PROJECT_ROOT)}")
    for part_dir in part_dirs:
        n = 0
        for hash_dir in part_dir.iterdir():
            if not hash_dir.is_dir():
                continue
            for flac in hash_dir.glob("*.flac"):
                key = f"{hash_dir.name}/{flac.name}"
                audio_index[key] = flac
                n += 1
        print(f"  {part_dir.name}: {n:,} files indexed")

    print(f"Total: {len(audio_index):,} FLAC files in index")
else:
    print("⚠️  cfg.audio_base_dir is empty — skipping audio indexing.")
    print("   Set it to data/unpacked/ParlaSpeech-{LANG}-audio and re-run.")

---

## 9. Convert FLAC → WAV

Look up each record's `audio` path in the index, convert to 16 kHz mono PCM WAV.
Set `r["_wav_path"]` (project-relative) for use in JSONL outputs.

`cleanup_flac=True` deletes each FLAC immediately after its WAV is written.

In [ ]:
import soundfile as sf
import numpy as np

wav_out_dir = PROJECT_ROOT / cfg.output_wav_dir
wav_out_dir.mkdir(parents=True, exist_ok=True)

n_done = n_skip = n_miss = n_err = 0

for r in tqdm(records, desc="FLAC→WAV", unit=" files"):
    raw = r.get("audio")  # e.g. "2iYgi9Ef6nM/2iYgi9Ef6nM_5686.7-5701.88.flac"
    if not raw:
        r["_wav_path"] = None
        n_miss += 1
        continue

    # Resolve WAV output path (same relative structure, .wav extension)
    wav_path = wav_out_dir / Path(raw).with_suffix(".wav")
    r["_wav_path"] = str(wav_path.relative_to(PROJECT_ROOT))

    if cfg.skip_existing_wavs and wav_path.exists():
        n_skip += 1
        continue

    if not audio_index:
        r["_wav_path"] = None
        n_miss += 1
        continue

    flac_path = audio_index.get(raw)
    if flac_path is None:
        r["_wav_path"] = None
        n_miss += 1
        continue

    try:
        wav_path.parent.mkdir(parents=True, exist_ok=True)
        data, sr = sf.read(str(flac_path), dtype="float32", always_2d=False)
        if data.ndim == 2:
            data = data.mean(axis=1)
        if sr != cfg.target_sample_rate:
            import librosa
            data = librosa.resample(data, orig_sr=sr, target_sr=cfg.target_sample_rate)
        sf.write(str(wav_path), data, cfg.target_sample_rate, subtype="PCM_16")

        if cfg.cleanup_flac:
            flac_path.unlink()

        n_done += 1
    except Exception as e:
        print(f"❌ {raw}: {e}")
        r["_wav_path"] = None
        n_err += 1

print(f"\nDone. converted={n_done}  skipped_existing={n_skip}  missing={n_miss}  errors={n_err}")
if n_miss > 0 and not audio_index:
    print("   (all misses because audio_index is empty — set cfg.audio_base_dir)")

---

## 10. Write instance JSONL

**`"utterance"` mode** — one line per utterance.
- `labels.filled_pause_present`: 1 if ≥1 FP in this utterance, else 0.
- `meta.fp_intervals`: the raw FP timestamps for reference (not a training label).

**`"fp_event"` mode** — one line per filled-pause event.
- `file_id`: `{utterance_id}_fp_{i}`
- `audio_path`: same full-utterance WAV (not cut; model sees whole clip)
- `start_t` / `end_t`: FP boundaries within the utterance
- `labels.filled_pause_present`: always 1
- Utterances with zero FPs are skipped entirely.

In [ ]:
import json

# TODO: currently overwrites the same file regardless of mode.
# Options: (a) auto-name by mode, (b) "both" mode writing two files in one pass.

if cfg.instance_mode == "utterance":
    def _make_records(r):
        fps = r.get("filled_pauses") or []
        si  = r.get("speaker_info", {})
        return [{
            "file_id":        r["id"],
            "audio_path":     r.get("_wav_path"),
            "audio_path_raw": r.get("audio"),
            "duration_s":     r.get("audio_length"),
            "text":           r.get("text"),
            "lang":           si.get("Lang", cfg.lang),
            "split":          r["_split"],
            "labels": {
                "filled_pause_present": 1 if fps else 0,
            },
            "meta": {
                "speaker_id":     si.get("Speaker_ID"),
                "speaker_gender": si.get("Speaker_gender"),
                "speaker_role":   si.get("Speaker_role"),
                "date":           si.get("Date"),
                "party":          si.get("Speaker_party"),
                "n_filled_pauses": len(fps),
                "fp_intervals":   [{"time_s": fp["time_s"], "time_e": fp["time_e"]} for fp in fps],
            },
        }]

elif cfg.instance_mode == "fp_event":
    def _make_records(r):
        fps = r.get("filled_pauses") or []
        si  = r.get("speaker_info", {})
        out = []
        for i, fp in enumerate(fps):
            out.append({
                "file_id":        f"{r['id']}_fp_{i}",
                "audio_path":     r.get("_wav_path"),
                "audio_path_raw": r.get("audio"),
                "duration_s":     r.get("audio_length"),
                "start_t":        fp["time_s"],   # FP start relative to utterance
                "end_t":          fp["time_e"],   # FP end relative to utterance
                "text":           r.get("text"),
                "lang":           si.get("Lang", cfg.lang),
                "split":          r["_split"],
                "labels": {
                    "filled_pause_present": 1,
                },
                "meta": {
                    "speaker_id":         si.get("Speaker_ID"),
                    "speaker_gender":     si.get("Speaker_gender"),
                    "speaker_role":       si.get("Speaker_role"),
                    "date":               si.get("Date"),
                    "party":              si.get("Speaker_party"),
                    "utterance_id":       r["id"],
                    "fp_index":           i,
                    "n_fps_in_utterance": len(fps),
                },
            })
        return out
else:
    raise ValueError(f"Unknown instance_mode {cfg.instance_mode!r}. Use 'utterance' or 'fp_event'.")

out_instance = PROJECT_ROOT / cfg.output_instance_jsonl
out_instance.parent.mkdir(parents=True, exist_ok=True)

n_lines = 0
with open(out_instance, "w", encoding="utf-8") as f:
    for r in records:
        for rec in _make_records(r):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n_lines += 1

print(f"✅ Wrote {n_lines} records → {out_instance.relative_to(PROJECT_ROOT)}")
print(f"   mode: {cfg.instance_mode}")

---

## 11. Write frame JSONL

One record per utterance. `labels.filled_pause` is a 50 Hz binary list.

In [ ]:
import json

out_frame = PROJECT_ROOT / cfg.output_frame_jsonl
out_frame.parent.mkdir(parents=True, exist_ok=True)

def _make_frame_record(r):
    fps = r.get("filled_pauses") or []
    dur = r.get("audio_length", 0)
    si  = r.get("speaker_info", {})
    return {
        "file_id":        r["id"],
        "audio_path":     r.get("_wav_path"),
        "audio_path_raw": r.get("audio"),
        "duration_s":     dur,
        "text":           r.get("text"),
        "lang":           si.get("Lang", cfg.lang),
        "split":          r["_split"],
        "frame_rate_hz":  cfg.frame_rate_hz,
        "labels": {
            "filled_pause": compute_frame_labels(fps, dur, cfg.frame_rate_hz),
        },
        "meta": {
            "speaker_id":     si.get("Speaker_ID"),
            "speaker_gender": si.get("Speaker_gender"),
            "speaker_role":   si.get("Speaker_role"),
            "date":           si.get("Date"),
            "party":          si.get("Speaker_party"),
        },
    }

with open(out_frame, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(_make_frame_record(r), ensure_ascii=False) + "\n")

print(f"✅ Wrote {len(records)} records → {out_frame.relative_to(PROJECT_ROOT)}")

---

## 12. Sanity checks

In [ ]:
import json

# 12a. Frame label length vs audio_length
mismatches = []
with open(out_frame, encoding="utf-8") as f:
    for line in f:
        r        = json.loads(line)
        dur      = r.get("duration_s") or 0
        expected = round(dur * cfg.frame_rate_hz)
        actual   = len(r["labels"]["filled_pause"])
        if abs(actual - expected) > 1:
            mismatches.append((r["file_id"], expected, actual))

if mismatches:
    print(f"⚠️  {len(mismatches)} frame-length mismatch(es):")
    for fid, exp, act in mismatches[:5]:
        print(f"  {fid}: expected ~{exp} got {act}")
else:
    print("✅ All frame label lengths match audio_length (±1 frame)")

# 12b. Instance label distribution
with open(out_instance, encoding="utf-8") as f:
    inst_records = [json.loads(l) for l in f]

n_pos = sum(1 for r in inst_records if r["labels"].get("filled_pause_present", 0) == 1)
n_tot = len(inst_records)
print(f"\nInstance JSONL ({cfg.instance_mode} mode): {n_tot} records")
if cfg.instance_mode == "utterance":
    print(f"  positive (FP present): {n_pos} ({100*n_pos/max(1,n_tot):.1f}%)")
    print(f"  negative:              {n_tot - n_pos}")
else:
    print(f"  all positive (fp_event mode) — {n_pos}/{n_tot}")

# 12c. WAV path coverage
n_null = sum(1 for r in inst_records if r.get("audio_path") is None)
print(f"\nWAV paths: {n_tot - n_null}/{n_tot} resolved  ({n_null} null)")
if n_null > 0:
    print("   → run cell 9 with cfg.audio_base_dir set to resolve remaining")

---

## Next

- **Chapter 2** — point `20_sniff_dataset.ipynb` at the frame or instance JSONL.
- **Chapter 4 (frame model)** — load `parlaspeech_{lang}_frame.jsonl`.
- Run another language by changing `cfg.lang` and re-running from cell 1.
- Switch `cfg.instance_mode` and re-run cells 10+ to regenerate the instance JSONL in the other flavor.

**Audio path resolution is the known blocker for training.** If `n_null > 0` above,
set `cfg.audio_base_dir = "data/unpacked/ParlaSpeech-{LANG}-audio"` and re-run cells 8–12.